# 01 · Оптический поток: девять управляемых стендов
К лекции 9. Здесь девять стендов: координаты и warping, наблюдаемость LK,
пирамида, регуляризация, масштаб поля, EPE и маски, реальные Farnebäck/DIS,
обратная согласованность и перенос информации между кадрами.
Работаем на CPU, без скачивания данных и весов.

**Порядок каждого блока:** короткая теория → стенд → задания под стендом.
В таблице опытов записывайте прогноз до изменения параметра, затем настройки,
численный результат и объяснение. Нужны минимум 18 опытов: по два на каждый
стенд, включая нормальный и трудный случай. Кнопка «Исходные настройки»
позволяет возвращаться к общей точке сравнения. Farnebäck/DIS рассчитываются
по кнопке; остальные стенды обновляются после отпускания ползунка.

**Соглашение:** forward flow f₁₂(x)=(u,v) задан на сетке I₁ и переносит точку
в x+f₁₂(x) кадра I₂. x направлен вправо, y вниз; единицы — пиксели между
двумя кадрами. Реконструкция: Î₁(x)=I₂(x+f₁₂(x)). Это чтение второго кадра
в найденной точке, а не смена знака массива. Backward flow f₂₁ живет на сетке
I₂: его нужно прочитать в x+f₁₂(x). Цвет показывает направление, яркость —
модуль с указанной шкалой; цветовая карта сама по себе не измеряет качество.

Учебные поля и точные синтетические преобразования явно отделены от выхода
алгоритмов. Числа демо не являются результатами на Sintel/KITTI.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'cv2': 'opencv-python-headless==4.12.0.88', 'ipywidgets': 'ipywidgets>=7.7,<9'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ('calcOpticalFlowFarneback', 'DISOpticalFlow_create', 'calcOpticalFlowPyrLK')
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import time,json,csv,hashlib,os
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi':110,'font.size':10})
assert hasattr(cv2,'calcOpticalFlowFarneback') and hasattr(cv2,'DISOpticalFlow_create')
print('OpenCV',cv2.__version__,'; CPU threads:',cv2.getNumThreads())

In [ ]:
def grid(h,w):
 y,x=np.mgrid[:h,:w];return x.astype(np.float32),y.astype(np.float32)

def texture(seed,h=128,w=192):
 rng=np.random.default_rng(seed);a=cv2.GaussianBlur(rng.normal(0,1,(h,w)).astype(np.float32),(0,0),.8)
 a=(a-a.mean())/(a.std()+1e-8)*35+127
 a=np.uint8(np.clip(a,0,255));a[10:35,12:52]=128
 return a

def flow_rgb(flow,vmax):
 u,v=flow[...,0].astype(np.float32),flow[...,1].astype(np.float32)
 mag,angle=cv2.cartToPolar(u,v)
 hsv=np.empty((*u.shape,3),np.uint8);hsv[...,0]=np.uint8(angle/(2*np.pi)*179);hsv[...,1]=255;hsv[...,2]=np.uint8(np.clip(mag/vmax,0,1)*255)
 return cv2.cvtColor(hsv,cv2.COLOR_HSV2RGB)

def draw_legend():
 x,y=grid(81,81);f=np.dstack([(x-40)/40,(y-40)/40]);rgb=flow_rgb(f,1);rgb[np.linalg.norm(f,axis=2)>1]=255
 fig,ax=plt.subplots(figsize=(3,3));ax.imshow(rgb);ax.set(xticks=[0,40,80],xticklabels=['−u','0','+u'],yticks=[0,40,80],yticklabels=['−v','0','+v'],title='Direction; brightness = magnitude');plt.show();plt.close(fig)

In [ ]:
import copy
import ipywidgets as W
from IPython.display import display
from IPython.utils.capture import capture_output
PANELS = {}

def panel(name, fn, controls):
    """Recompute cheap stands on release; dense estimation uses an explicit button."""
    output = W.Output(layout=W.Layout(width='100%'))
    manual = name == 'dense'
    initial = {key: copy.deepcopy(control.value) for key, control in controls.items()}
    state = {'calls': 0, 'last': None, 'dirty': False, 'suspended': False}
    notice = W.HTML()

    def refresh(change=None):
        with capture_output(display=True) as captured:
            result = fn(**{key: control.value for key, control in controls.items()})
        items = [{'output_type': 'display_data', 'data': item.data, 'metadata': item.metadata}
                 for item in captured.outputs]
        for kind in ('stdout', 'stderr'):
            if getattr(captured, kind):
                items.append({'output_type': 'stream', 'name': kind,
                              'text': getattr(captured, kind)})
        output.outputs = tuple(items)
        state.update(last=result, calls=state['calls'] + 1, dirty=False)
        notice.value = '<small>Показан результат текущих настроек.</small>'

    def changed(change):
        if state['suspended']:
            return
        state['dirty'] = True
        if manual:
            notice.value = '<b>Настройки изменены. Нажмите «Рассчитать» для нового результата.</b>'
        else:
            refresh()

    def reset(change=None):
        state['suspended'] = True
        try:
            for key, value in initial.items():
                controls[key].value = copy.deepcopy(value)
        finally:
            state['suspended'] = False
        refresh()

    for control in controls.values():
        if hasattr(control, 'continuous_update'):
            control.continuous_update = False
        control.layout = W.Layout(width='330px', margin='2px 10px 2px 0')
        if hasattr(control, 'style') and hasattr(control.style, 'description_width'):
            control.style.description_width = '160px'
        control.observe(changed, names='value')
    button = W.Button(description='Рассчитать' if manual else 'Повторить опыт', icon='refresh')
    reset_button = W.Button(description='Исходные настройки', icon='undo')
    button.on_click(refresh)
    reset_button.on_click(reset)
    widget = W.VBox([
        W.Box(list(controls.values()), layout=W.Layout(display='flex', flex_flow='row wrap')),
        W.HBox([button, reset_button]), notice, output
    ], layout=W.Layout(width='100%'))
    PANELS[name] = {'controls': controls, 'state': state, 'refresh': refresh,
                    'button': button, 'reset': reset_button, 'output': output,
                    'widget': widget, 'function': fn, 'initial': initial, 'manual': manual}
    display(widget)
    refresh()

In [ ]:
def core_mean(values, mask):
    return float(np.asarray(values)[mask].mean()) if np.any(mask) else float('nan')

def core_warp(image, flow, interpolation=cv2.INTER_LINEAR, border=cv2.BORDER_CONSTANT):
    h,w=flow.shape[:2];x,y=grid(h,w)
    mx=x+flow[...,0];my=y+flow[...,1]
    inside=(mx>=0)&(mx<=w-1)&(my>=0)&(my<=h-1)
    return cv2.remap(np.asarray(image,np.float32),mx.astype(np.float32),my.astype(np.float32),
                     interpolation,borderMode=border),inside

def core_slider(value, low, high, description, step=1):
    kind=W.IntSlider if all(isinstance(v,int) for v in (value,low,high,step)) else W.FloatSlider
    return kind(value=value,min=low,max=high,step=step,description=description,
                continuous_update=False,style={'description_width':'initial'})

In [ ]:
def adv_mean(values, mask):
    return float(np.asarray(values)[mask].mean()) if np.any(mask) else float('nan')

def adv_image(seed=61, h=96, w=144, pattern='texture'):
    x, y = grid(h, w)
    if pattern == 'stripes':
        return np.uint8(127 + 65 * np.sin(x * .5))
    if pattern == 'flat':
        return np.full((h, w), 127, np.uint8)
    return texture(seed, h, w)

def adv_pair(dx, dy=2, kind='translation', brightness=0., noise=0., h=96, w=144):
    a = adv_image(61, h, w)
    x, y = grid(h, w)
    matrix = np.float32([[1, 0, dx], [0, 1, dy]])
    gt = np.zeros((h, w, 2), np.float32)
    if kind == 'translation':
        gt[:] = [dx, dy]
        b = cv2.warpAffine(a, matrix, (w, h), flags=cv2.INTER_LINEAR)
        valid = (x+dx >= 0) & (x+dx <= w-1) & (y+dy >= 0) & (y+dy <= h-1)
        boundary = (x < 8) | (x >= w-8) | (y < 8) | (y >= h-8)
    else:
        obj = np.zeros((h, w), np.uint8)
        obj[h//3:2*h//3, w//3:2*w//3] = 1
        foreground = adv_image(1061, h, w)
        a = a.copy(); a[obj > 0] = foreground[obj > 0]
        moved_mask = cv2.warpAffine(obj, matrix, (w, h), flags=cv2.INTER_NEAREST) > 0
        source = np.where(obj > 0, foreground, 0).astype(np.uint8)
        moved = cv2.warpAffine(source, matrix, (w, h), flags=cv2.INTER_NEAREST)
        b = adv_image(61, h, w); b[moved_mask] = moved[moved_mask]
        gt[obj > 0] = [dx, dy]
        valid = ((obj > 0) | ~moved_mask) & (x+gt[..., 0] >= 0) & (x+gt[..., 0] <= w-1) & (y+gt[..., 1] >= 0) & (y+gt[..., 1] <= h-1)
        kernel = np.ones((7, 7), np.uint8)
        boundary = (cv2.dilate(obj, kernel) > 0) & (cv2.erode(obj, kernel) == 0)
    perturbation = np.random.default_rng(62).normal(0, noise, b.shape)
    b = np.uint8(np.clip(b.astype(float) + brightness + perturbation, 0, 255))
    return a, b, gt, valid, boundary

def adv_warp(image2, flow):
    h, w = flow.shape[:2]; x, y = grid(h, w)
    mx = x + flow[..., 0]; my = y + flow[..., 1]
    inside = (mx >= 0) & (mx <= w-1) & (my >= 0) & (my <= h-1)
    sampled = cv2.remap(np.asarray(image2, np.float32), mx.astype(np.float32), my.astype(np.float32), cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return sampled, inside

def adv_resize_flow(flow, target):
    h, w = flow.shape[:2]; nh, nw = target
    result = cv2.resize(flow, (nw, nh), interpolation=cv2.INTER_LINEAR)
    result[..., 0] *= nw/w; result[..., 1] *= nh/h
    return result

In [ ]:
draw_legend()

## 1. Направление потока, реконструкция и координаты

Forward flow f₁₂(x)=(u,v) задан на сетке I₁ и переносит точку в x+f₁₂(x) кадра I₂. Реконструкция Î₁(x)=I₂(x+f₁₂(x)) читает второй кадр по forward-полю. На I₁ стрелка показывает истинный перенос; крест на I₂ — выбранную координату чтения. Ползунки точки позволяют проследить конкретное соответствие, включая выход за границу.

Маска для сравнения знаков одна: обе координаты x±f должны быть внутри изображения. Продолжение отражением даёт численное значение вне кадра, но не наблюдение. Ненулевой фотометрический residual возможен при правильном потоке из-за изменения яркости и интерполяции; это не EPE.

In [ ]:
def show_direction(dx,dy,sign,brightness,interpolation,border,probe_x,probe_y):
    h,w=72,112;a=texture(4,h,w).astype(np.float32)
    interp={'linear':cv2.INTER_LINEAR,'nearest':cv2.INTER_NEAREST}[interpolation]
    bord={'constant':cv2.BORDER_CONSTANT,'reflect':cv2.BORDER_REFLECT_101}[border]
    b=cv2.warpAffine(a,np.float32([[1,0,dx],[0,1,dy]]),(w,h),flags=interp,
                     borderMode=cv2.BORDER_CONSTANT)+brightness
    f=np.empty((h,w,2),np.float32);f[:]=[sign*dx,sign*dy]
    reconstructed,inside=core_warp(b,f,interp,bord)
    x,y=grid(h,w)
    # Same evaluation pixels for BOTH signs; border extrapolation is never GT.
    valid=(x+dx>=0)&(x+dx<=w-1)&(y+dy>=0)&(y+dy<=h-1)&(x-dx>=0)&(x-dx<=w-1)&(y-dy>=0)&(y-dy<=h-1)
    residual=np.abs(a-reconstructed);mae=core_mean(residual,valid)
    px=int(probe_x);py=int(probe_y);qx=px+sign*dx;qy=py+sign*dy
    fig,axs=plt.subplots(2,2,figsize=(11,6.2),constrained_layout=True)
    for ax,img,title in zip(axs.flat,[a,b,reconstructed,np.where(valid,residual,np.nan)],
       ['I₁: source point', 'I₂: known translation + brightness',
        'Reconstruction I₂(grid ± flow)',f'Common-valid residual; MAE={mae:.3f}']):
        im=ax.imshow(img,cmap='magma' if ax is axs[1,1] else 'gray',
                     vmin=0,vmax=80 if ax is axs[1,1] else 255)
        ax.set_title(title);ax.set(xticks=[],yticks=[])
    axs[0,0].plot(px,py,'ro',ms=5)
    axs[0,0].quiver([px],[py],[dx],[dy],angles='xy',scale_units='xy',scale=1,color='red')
    axs[0,1].plot([qx],[qy],'rx',ms=8)
    axs[1,0].plot(px,py,'ro',ms=5)
    fig.colorbar(im,ax=axs[1,1],shrink=.8,label='intensity difference')
    plt.show();plt.close(fig)
    probe={'source':[px,py],'sample':[float(qx),float(qy)],
           'inside':bool(inside[py,px]),'residual':float(residual[py,px]),
           'source_intensity':float(a[py,px]),'reconstructed_intensity':float(reconstructed[py,px])}
    print(f'f₁₂=({dx:g},{dy:g}) px on I₁. Sampling sign={sign:+d}; valid count={valid.sum()}; MAE={mae:.4f}.')
    print('Probe:',probe)
    print('Border policy changes extrapolated pixels, but never makes them valid. Brightness and subpixel interpolation can leave residual even at exact flow.')
    return {'mae':mae,'count':int(valid.sum()),'probe':probe,'all_mae':float(residual.mean())}
panel('direction',show_direction,{
    'dx':core_slider(4.,-12.,12.,'u, px',.5),
    'dy':core_slider(2.,-10.,10.,'v, px',.5),
    'sign':W.Dropdown(options=[('grid + flow',1),('grid − flow',-1)],value=1,description='Sampling'),
    'brightness':core_slider(0,-50,50,'Brightness Δ',10),
    'interpolation':W.Dropdown(options=['linear','nearest'],value='linear',description='Interpolation'),
    'border':W.Dropdown(options=['constant','reflect'],value='constant',description='Outside image'),
    'probe_x':core_slider(55,0,111,'Probe x'),
    'probe_y':core_slider(42,0,71,'Probe y')})

### Проверка на стенде 1

1. При u=4, v=2, Δ яркости=0 запишите прогноз для двух знаков семплирования. Переключите знак, сравните MAE и координату креста; верните правильный знак.
2. Перенесите пробу к краю, затем сравните constant/reflect. Какие значения изменились вне кадра и почему число общих valid-пикселей осталось прежним? Повторите с отрицательными u и v.
3. При правильном знаке добавьте яркость +30. Затем сравните линейную и ближайшую интерполяцию при u=4.5. Объясните, почему остаточная ошибка изображения не доказывает ошибку GT-потока.
4. Установите u=v=0 и Δ яркости=0. Почему проверка одного этого состояния не обнаруживает неправильный знак? Для двух опытов сохраните параметры, численный прогноз, MAE и объяснение.

## 2. Lucas–Kanade: окно, градиенты и наблюдаемость

В окне LK предполагается единый вектор: Iₓu+Iᵧv+Iₜ=0. Система решается по наблюдаемым градиентам, а не по названию рисунка. Здесь Iₜ строится точно из линейной системы и управляемого шума; это проверяемая модель локальной линеаризации, а не запуск полного трекера. Красная рамка задаёт используемое окно.

Карта показывает стоимость только данных; вытянутая долина означает неоднозначность. Собственные значения вычисляются по AᵀA/N **до ridge**. Контраст масштабирует информацию, окно меняет состав наблюдений, угол поворачивает наблюдаемую компоненту. Регуляризация не добавляет недостающего направления градиента.

In [ ]:
def show_aperture(pattern,angle,window,contrast,noise,ridge,true_u,true_v):
    h=w=61;yy,xx=np.mgrid[-30:31,-30:31];theta=np.deg2rad(angle)
    z=(xx*np.cos(theta)+yy*np.sin(theta))/7
    q=(-xx*np.sin(theta)+yy*np.cos(theta))/7
    if pattern=='edge':
        patch=contrast*z;dz=np.full(z.shape,contrast/7);dq=np.zeros_like(z)
    elif pattern=='corner':
        patch=contrast*(np.tanh(z)+np.tanh(q))
        dz=contrast*(1-np.tanh(z)**2)/7;dq=contrast*(1-np.tanh(q)**2)/7
    else:patch=np.zeros_like(z);dz=np.zeros_like(z);dq=np.zeros_like(z)
    gx=dz*np.cos(theta)-dq*np.sin(theta);gy=dz*np.sin(theta)+dq*np.cos(theta)
    half=window//2;roi=(np.abs(xx)<=half)&(np.abs(yy)<=half)
    A=np.c_[gx[roi],gy[roi]];true=np.array([true_u,true_v],float)
    # Fixed full-size noise: changing window changes which observations are used.
    disturbance=np.random.default_rng(40).normal(0,noise,(h,w))
    It=-(A@true)+disturbance[roi]
    M=A.T@A/len(A);rhs=-A.T@It/len(A)
    estimate=np.linalg.lstsq(M+ridge*np.eye(2),rhs,rcond=None)[0]
    eigen=np.maximum(np.linalg.eigvalsh(M),0)
    observable=bool(eigen.min()>1e-6)
    bound=max(4.,float(np.max(np.abs(estimate)))*1.15)
    values=np.linspace(-bound,bound,81);uu,vv=np.meshgrid(values,values)
    trial=np.stack([uu,vv],axis=-1)
    # Normal-equation form avoids a large trial x observation array.
    objective=np.einsum('...i,ij,...j->...',trial,M,trial)-2*np.einsum('...i,i->...',trial,rhs)+np.mean(It**2)
    fig,axs=plt.subplots(2,2,figsize=(10.8,6.2),constrained_layout=True)
    axs[0,0].imshow(patch,cmap='gray');axs[0,0].add_patch(plt.Rectangle((30-half-.5,30-half-.5),window,window,fill=False,color='tomato',lw=2))
    axs[0,0].set_title(f'{pattern}: {window}×{window} observation window');axs[0,0].axis('off')
    skip=5;axs[0,1].quiver(xx[::skip,::skip],yy[::skip,::skip],gx[::skip,::skip],gy[::skip,::skip],angles='xy')
    axs[0,1].set(title='Measured gradient directions',aspect='equal');axs[0,1].invert_yaxis()
    im=axs[1,0].imshow(np.maximum(objective,0),origin='lower',extent=(-bound,bound,-bound,bound),cmap='magma',aspect='equal')
    axs[1,0].plot(*true,'c*',ms=12,label='true');axs[1,0].plot(*estimate,'wx',ms=9,label='estimate')
    axs[1,0].set(title='Unregularized data cost',xlabel='u, px',ylabel='v, px');axs[1,0].legend(fontsize=8)
    fig.colorbar(im,ax=axs[1,0],shrink=.75)
    axs[1,1].bar(np.arange(2)-.18,true,width=.36,label='true')
    axs[1,1].bar(np.arange(2)+.18,estimate,width=.36,label='estimate');axs[1,1].set(xticks=[0,1],xticklabels=['u','v'],ylabel='px',title=f'EPE={np.linalg.norm(estimate-true):.3f}')
    axs[1,1].legend();plt.show();plt.close(fig)
    print('Eigenvalues of AᵀA/N BEFORE ridge:',eigen,'; observable:',observable)
    print('Estimated flow:',estimate,'; condition:',float(eigen.max()/eigen.min()) if eigen.min()>1e-12 else 'singular', '; observations:',len(A))
    print('It is generated by the exact linear system Ix·u+Iy·v+It=0 with controlled noise. Ridge stabilizes a solution; it does not supply an unseen gradient direction.')
    return {'estimate':estimate.tolist(),'eigenvalues':eigen.tolist(),'min_eigen':float(eigen.min()),
            'observable':observable,'epe':float(np.linalg.norm(estimate-true)),'observations':len(A)}
panel('aperture',show_aperture,{
    'pattern':W.Dropdown(options=['edge','corner','flat'],value='edge',description='Patch'),
    'angle':core_slider(0,0,180,'Gradient angle, °',15),
    'window':W.SelectionSlider(options=[7,15,21,31,45,61],value=21,description='Window',continuous_update=False),
    'contrast':core_slider(1.,0.,3.,'Contrast',.25),
    'noise':core_slider(0.,0.,.5,'It noise',.025),
    'ridge':core_slider(0.,0.,.1,'Ridge',.005),
    'true_u':core_slider(2.,-3.,3.,'True u',.5),
    'true_v':core_slider(1.,-3.,3.,'True v',.5)})

### Проверка на стенде 2

1. Для edge, угла 0°, шума=ridge=0 и true=(2,1) предскажите решение и ранг. Затем поверните градиент на 90°: какая компонента теперь восстанавливается?
2. Переключите corner и flat, измените размер окна. Сопоставьте форму стоимости, оба собственных значения и ошибку; не объясняйте результат лишь количеством пикселей.
3. На corner уменьшите контраст, добавьте фиксированный шум и сравните два окна. Затем добавьте ridge: где уменьшается разброс и где появляется смещение к нулю?
4. На edge измените true-вектор вдоль ненаблюдаемого направления. Может ли измеренная система различить эти движения? Зафиксируйте прогноз, решение и собственные значения минимум для двух опытов.

## 3. Пирамида и предел локального поиска LK

Локальная линеаризация удобна для малого перемещения. В пирамиде тот же перенос в $2^L$ раз меньше на уровне $L$; оценку уточняют при возврате к исходному разрешению. Здесь выполняется настоящий `calcOpticalFlowPyrLK`: одинаковые точки, окна и критерий остановки, один уровень против заданной пирамиды. Показан и фактически построенный уровень: размер изображения и окна может ограничить глубину.

Точка в текстуре, внутри плоской зоны и на ее границе получает разную информацию. Успешный статус LK не равен доказанной точности: проверяем GT ошибки и число принятых точек вместе. Шум добавлен только во второй кадр; GT перенос от этого не меняется.

In [ ]:
def show_pyramid(dx, dy, levels, window, iterations, pattern, noise, probe):
    h, w = 128, 192
    a = adv_image(71, h, w, pattern)
    b = cv2.warpAffine(a, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h))
    b = np.uint8(np.clip(b.astype(float) + np.random.default_rng(72).normal(0, noise, b.shape), 0, 255))
    yy, xx = np.mgrid[40:105:24, 64:153:28]
    points = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    probe_points = {'texture': [94, 68], 'flat': [30, 22], 'edge': [52, 22]}
    points = np.vstack([points, np.float32(probe_points[probe])]).reshape(-1, 1, 2)
    target = points[:, 0, :] + [dx, dy]
    truth_valid = (target[:, 0] >= 0) & (target[:, 0] <= w-1) & (target[:, 1] >= 0) & (target[:, 1] <= h-1)
    tracks = []
    for requested in [0, levels]:
        predicted, status, _ = cv2.calcOpticalFlowPyrLK(a, b, points, None,
            winSize=(window, window), maxLevel=requested,
            criteria=(cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, iterations, .001))
        predicted = predicted[:, 0, :]
        good = status.ravel().astype(bool) & truth_valid & np.isfinite(predicted).all(axis=1)
        error = np.linalg.norm(predicted-target, axis=1)
        built, _ = cv2.buildOpticalFlowPyramid(a, (window, window), requested, withDerivatives=False)
        tracks.append((predicted, good, error, int(built)))
    fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
    for ax, (predicted, good, error, actual), label in zip(axes[0], tracks, ['Single level', 'Pyramid']):
        ax.imshow(a, cmap='gray', vmin=0, vmax=255)
        p = points[:, 0, :]
        ax.quiver(p[good, 0], p[good, 1], predicted[good, 0]-p[good, 0], predicted[good, 1]-p[good, 1], angles='xy', scale_units='xy', scale=1, color='tomato')
        ax.scatter(target[:, 0], target[:, 1], s=24, facecolors='none', edgecolors='lime', label='True target')
        ax.scatter(*p[-1], color='yellow', marker='x', s=90)
        ax.set_title(f'{label}: max level actually built {actual}; accepted {good.sum()}/{len(good)}'); ax.axis('off')
    axes[0, 0].legend(loc='upper right', fontsize=8)
    p = points[-1, 0]; radius = window//2
    axes[1, 0].imshow(a, cmap='gray', vmin=0, vmax=255)
    axes[1, 0].set(xlim=(p[0]-radius-2, p[0]+radius+2), ylim=(p[1]+radius+2, p[1]-radius-2), title=f'Probe window: {probe}, {window}×{window}')
    for k, ((_, good, error, _), label) in enumerate(zip(tracks, ['Single level', 'Pyramid'])):
        axes[1, 1].bar(np.arange(len(error)) + (k-.5)*.38, np.where(good, error, np.nan), width=.38, label=label)
    axes[1, 1].set(xlabel='Point index; last = selected probe', ylabel='Endpoint error, original px', title='Only LK-accepted, GT-visible points'); axes[1, 1].legend()
    plt.show(); plt.close(fig)
    result = {'single_epe': adv_mean(tracks[0][2], tracks[0][1]), 'pyramid_epe': adv_mean(tracks[1][2], tracks[1][1]),
        'single_accepted': int(tracks[0][1].sum()), 'pyramid_accepted': int(tracks[1][1].sum()),
        'probe_accepted': bool(tracks[1][1][-1]), 'probe_epe': float(tracks[1][2][-1]) if tracks[1][1][-1] else float('nan'),
        'probe_flow': (tracks[1][0][-1]-points[-1, 0]).tolist(), 'actual_level': tracks[1][3],
        'coarse_displacement': [dx/(2**tracks[1][3]), dy/(2**tracks[1][3])], 'point_count': len(points)}
    print(result)
    print('Зеленые окружности — GT; красные стрелки — настоящий OpenCV PyrLK. Status=1 означает сходимость проверки LK, а не доказанную точность. Пустая группа → EPE NaN. Грубый уровень уменьшает видимое смещение, но теряет текстуру.')
    return result

panel('pyramid', show_pyramid, {
    'dx': W.IntSlider(value=12, min=-24, max=24, description='Сдвиг x'),
    'dy': W.IntSlider(value=2, min=-12, max=12, description='Сдвиг y'),
    'levels': W.IntSlider(value=3, min=0, max=3, description='Max level'),
    'window': W.SelectionSlider(options=[7, 15, 25, 35], value=15, description='Окно LK'),
    'iterations': W.IntSlider(value=20, min=1, max=40, description='Итерации'),
    'pattern': W.Dropdown(options=[('Текстура', 'texture'), ('Полосы', 'stripes'), ('Без текстуры', 'flat')], value='texture', description='Изображение'),
    'noise': W.IntSlider(value=0, min=0, max=25, description='Шум I2'),
    'probe': W.Dropdown(options=[('Текстура', 'texture'), ('Плоская зона', 'flat'), ('Граница зоны', 'edge')], value='texture', description='Пробная точка')})

### Проверка на стенде 3

1. **Прогноз → проверка:** при сдвиге x=12 сравните один уровень и пирамиду. Запишите EPE **и число** принятых точек. Затем задайте max level=0: два режима должны совпасть.
2. Увеличьте сдвиг до 24. Меняйте только глубину 0 → 3, затем окно 7 → 35. Сопоставьте реально построенный уровень и перенос на нем; почему большое окно не всегда помогает?
3. При нулевом шуме выберите пробную точку «Плоская зона», затем «Граница зоны»; измените окно. Где появился второй градиент? Сравните с изображением из полос и с полностью плоским изображением.
4. Добавьте шум, затем увеличьте число итераций 1 → 40. Объясните случай, когда больше итераций не исправляет неверное соответствие. Приведите пример статуса LK=1 с заметной EPE.

## 4. Horn–Schunck: итерации и цена гладкости

Horn–Schunck связывает оценки соседних пикселей: энергия содержит ошибку Iₓu+Iᵧv+Iₜ и штраф α²(|∇u|²+|∇v|²). Стенд действительно обновляет поле итерациями с фиксированными производными изображений и усреднением восьми соседей. Это одноуровневая линейная версия; больших перемещений без пирамиды она не обязана восстанавливать.

В translation движется всё изображение; в moving_patch текстурная вставка движется над неподвижным фоном. Закрытый фон исключён по известной геометрии; GT-valid маска не зависит от α. Показываются истинное и найденное u, ошибка, ход итераций, внутренние и граничные EPE. Фотометрический residual и линейная невязка имеют иной смысл, чем EPE.

In [ ]:
def core_hs_pair(scene,shift,noise,flat_fraction,brightness,seed):
    h,w=72,112;x,y=grid(h,w);rng=np.random.default_rng(seed)
    bg=texture(seed,h,w).astype(np.float32)/255
    fg=texture(seed+100,h,w).astype(np.float32)/255
    flat_w=round(w*flat_fraction)
    if flat_w:
        bg[24:48,w//2-flat_w//2:w//2+(flat_w+1)//2]=.5
        fg[24:48,w//2-flat_w//2:w//2+(flat_w+1)//2]=.5
    gt=np.zeros((h,w,2),np.float32)
    if scene=='translation':
        a=bg;gt[...,0]=shift
        b=cv2.warpAffine(a,np.float32([[1,0,shift],[0,1,0]]),(w,h))
        valid=(x+shift>=0)&(x+shift<=w-1);boundary=(x<6)|(x>=w-6)
    else:
        obj=np.zeros((h,w),np.uint8);obj[18:54,32:80]=1
        a=bg.copy();a[obj>0]=fg[obj>0];gt[obj>0,0]=shift
        moved_mask=cv2.warpAffine(obj,np.float32([[1,0,shift],[0,1,0]]),(w,h),flags=cv2.INTER_NEAREST)>0
        moved=cv2.warpAffine(fg,np.float32([[1,0,shift],[0,1,0]]),(w,h))
        b=bg.copy();b[moved_mask]=moved[moved_mask]
        valid=((obj>0)|(~moved_mask))&(x+gt[...,0]>=0)&(x+gt[...,0]<=w-1)
        boundary=(cv2.dilate(obj,np.ones((9,9),np.uint8))>0)&(cv2.erode(obj,np.ones((9,9),np.uint8))==0)
    b=b+brightness+rng.normal(0,noise,(h,w)).astype(np.float32)
    # The same GT mask is used regardless of alpha, noise or iterations.
    return a.astype(np.float32),b.astype(np.float32),gt,valid,boundary

def show_smoothing(scene,shift,alpha,iterations,noise,flat_fraction,brightness,seed):
    a,b,gt,valid,boundary=core_hs_pair(scene,shift,noise,flat_fraction,brightness,seed)
    # Single-scale Horn–Schunck: data derivatives remain fixed during iteration.
    gx=(cv2.Sobel(a,cv2.CV_32F,1,0,ksize=3)+cv2.Sobel(b,cv2.CV_32F,1,0,ksize=3))/16
    gy=(cv2.Sobel(a,cv2.CV_32F,0,1,ksize=3)+cv2.Sobel(b,cv2.CV_32F,0,1,ksize=3))/16
    It=b-a;u=np.zeros_like(a);v=np.zeros_like(a)
    kernel=np.array([[1,2,1],[2,0,2],[1,2,1]],np.float32)/12
    denom=alpha**2+gx*gx+gy*gy;trace=[]
    for k in range(iterations+1):
        if k in {0,iterations} or k%5==0:
            error=np.sqrt((u-gt[...,0])**2+(v-gt[...,1])**2)
            trace.append((k,core_mean(error,valid),float(np.sqrt(np.mean((gx*u+gy*v+It)**2)))))
        if k==iterations:break
        ub=cv2.filter2D(u,-1,kernel,borderType=cv2.BORDER_REFLECT_101)
        vb=cv2.filter2D(v,-1,kernel,borderType=cv2.BORDER_REFLECT_101)
        correction=(gx*ub+gy*vb+It)/denom
        u=ub-gx*correction;v=vb-gy*correction
    flow=np.dstack([u,v]);epe=np.linalg.norm(flow-gt,axis=2)
    reconstructed,inside=core_warp(b,flow)
    photometric=core_mean(np.abs(a-reconstructed),valid&inside)*255
    edge=valid&boundary;interior=valid&~boundary
    fig,axs=plt.subplots(2,3,figsize=(12,6),constrained_layout=True)
    images=[a,b,gt[...,0],u,np.where(valid,epe,np.nan)]
    titles=['I₁','I₂: shifted layer + disturbance','True u, px',f'HS u after {iterations} iterations','GT-valid EPE, px']
    for ax,img,title in zip(axs.flat,images,titles):
        im=ax.imshow(img,cmap='gray' if ax in list(axs.flat)[:2] else 'viridis')
        ax.set_title(title);ax.axis('off')
        if ax not in list(axs.flat)[:2]:fig.colorbar(im,ax=ax,shrink=.7)
    t=np.asarray(trace);axs[1,2].plot(t[:,0],t[:,1],'o-',label='GT EPE, px')
    axs[1,2].plot(t[:,0],t[:,2]*255,'s-',label='linear residual ×255')
    axs[1,2].set(title='Two different diagnostics',xlabel='iteration');axs[1,2].legend(fontsize=8);axs[1,2].grid(alpha=.3)
    plt.show();plt.close(fig)
    result={'epe':core_mean(epe,valid),'boundary':core_mean(epe,edge),'interior':core_mean(epe,interior),
            'photometric_mae':photometric,'linear_residual':trace[-1][2],
            'count':int(valid.sum()),'iterations':iterations,'trace':trace,
            'max_flow':float(np.max(np.linalg.norm(flow,axis=2)))}
    print(result)
    print('Single-scale linearized Horn–Schunck with fixed derivatives and eight-neighbour averaging. Large displacement, occlusion and brightness changes violate its assumptions. Lower residual need not mean lower EPE.')
    return result
panel('smoothing',show_smoothing,{
    'scene':W.Dropdown(options=['translation','moving_patch'],value='moving_patch',description='Scene'),
    'shift':core_slider(1,-4,4,'True horizontal shift'),
    'alpha':W.SelectionSlider(options=[.01,.03,.1,.3,1.],value=.1,description='α',continuous_update=False),
    'iterations':core_slider(60,0,200,'Iterations',10),
    'noise':core_slider(0.,0.,.08,'I₂ noise',.01),
    'flat_fraction':core_slider(0.,0.,.6,'Flat region width',.1),
    'brightness':core_slider(0.,-.2,.2,'I₂ brightness Δ',.025),
    'seed':core_slider(9,1,20,'Texture seed')})

### Проверка на стенде 4

1. Для translation, сдвига 1, без шума и изменения яркости сравните 0, 60 и 200 итераций. Запишите обе диагностики: одинаково ли они меняются?
2. На moving_patch измените α .03 → .3 → 1 при одном seed. Сравните EPE на границе и внутри; где требование гладкости противоречит истинному полю?
3. Добавьте шум, затем увеличьте плоскую область при том же α. Как соседи помогают заполнить область без локальных градиентов и почему это не новое наблюдение движения?
4. Сравните сдвиги 1 и 4, затем отдельно добавьте изменение яркости. Что нарушает линеаризацию и постоянство яркости? Поменяйте seed и проверьте, сохраняется ли вывод; не подменяйте объяснение одним красивым полем.

## 5. Масштаб сетки, единицы векторов и потеря границы

При переносе поля H×W → H′×W′ меняются и сетка, и единицы компонентов: u′=u·W′/W, v′=v·H′/H. Интерполяция массива выполняет только первую часть. Стенд показывает настоящее пространственное поле поверх текстуры и намеренные ошибки масштабирования. Стрелки всегда выражены в пикселях показанного изображения.

Обратный перенос выполняется с правильными коэффициентами. Даже правильный путь туда и обратно может сгладить границу двух движений: это потеря при дискретизации, отдельно от ошибки единиц. «Unit error before return» сравнивает коэффициенты на уменьшенной сетке; EPE рисунка относится к сетке, указанной над ним.

In [ ]:
def show_resize(sx,sy,field_mode,interpolation,scale_mode,base_u,base_v,roundtrip):
    h,w=64,96;x,y=grid(h,w)
    field=np.zeros((h,w,2),np.float32);field[:]=[base_u,base_v]
    if field_mode=='two_motions':field[:,w//2:,0]*=-1;field[:,w//2:,1]*=-1
    elif field_mode=='rotation':field[...,0]+=-(y-(h-1)/2)*.12;field[...,1]+=(x-(w-1)/2)*.12
    nh,nw=round(h*sy),round(w*sx);effective=np.array([nw/w,nh/h])
    interp={'linear':cv2.INTER_LINEAR,'nearest':cv2.INTER_NEAREST}[interpolation]
    sampled=cv2.resize(field,(nw,nh),interpolation=interp)
    expected=sampled*effective
    factors={'correct':effective,'none':np.ones(2),'same_for_both':np.repeat(effective[0],2),'swapped_axes':effective[::-1]}[scale_mode]
    actual=sampled*factors;unit_error=float(np.linalg.norm(actual-expected,axis=2).mean())
    if roundtrip:
        actual=cv2.resize(actual,(w,h),interpolation=interp)/effective
        expected=field.copy();target=texture(4,h,w)
    else:target=cv2.resize(texture(4,h,w),(nw,nh),interpolation=interp)
    error=np.linalg.norm(actual-expected,axis=2);th,tw=actual.shape[:2]
    fig,axs=plt.subplots(2,2,figsize=(10.8,6),constrained_layout=True)
    def vectors(ax,image,f,title):
        ih,iw=f.shape[:2];xx,yy=grid(ih,iw);step=max(4,min(ih,iw)//8)
        ax.imshow(image,cmap='gray',vmin=0,vmax=255)
        ax.quiver(xx[::step,::step],yy[::step,::step],f[::step,::step,0],f[::step,::step,1],
                  angles='xy',scale_units='xy',scale=1,color='tomato')
        ax.set_title(title);ax.set(xlim=(-.5,iw-.5),ylim=(ih-.5,-.5));ax.axis('off')
    vectors(axs[0,0],texture(4,h,w),field,f'Original {w}×{h}: vector units = source px')
    vectors(axs[0,1],target,expected,f'Expected field on {tw}×{th} grid')
    vectors(axs[1,0],target,actual,f'Actual field: {scale_mode}'+(' → round trip' if roundtrip else ''))
    im=axs[1,1].imshow(error,cmap='magma',vmin=0,vmax=max(1,float(error.max())))
    axs[1,1].set_title(f'EPE on shown grid = {error.mean():.4f} px');axs[1,1].axis('off');fig.colorbar(im,ax=axs[1,1],shrink=.8)
    plt.show();plt.close(fig)
    probe=actual[th//3,tw//3]
    print(f'Grid: ({h},{w}) → ({nh},{nw}); actual component factors={factors}; required={effective}.')
    print(f'Unit error before return={unit_error:.5f} px on resized grid; shown mean EPE={error.mean():.5f}. Probe vector={probe}.')
    print('Round trip with correct units can still blur a motion boundary. Each component uses its own axis scale; arrows remain in shown-image pixels.')
    return {'vector':probe.tolist(),'epe':float(error.mean()),'unit_epe':unit_error,
            'target_shape':[th,tw],'effective_scales':effective.tolist(),'u_variation':float(np.std(actual[...,0]))}
panel('resize',show_resize,{
    'sx':W.SelectionSlider(options=[.25,.5,1.,1.5,2.],value=.5,description='Width scale',continuous_update=False),
    'sy':W.SelectionSlider(options=[.25,.5,1.,1.5,2.],value=1.5,description='Height scale',continuous_update=False),
    'field_mode':W.Dropdown(options=['constant','two_motions','rotation'],value='constant',description='Field'),
    'interpolation':W.Dropdown(options=['linear','nearest'],value='linear',description='Interpolation'),
    'scale_mode':W.Dropdown(options=['correct','none','same_for_both','swapped_axes'],value='correct',description='Component scaling'),
    'base_u':core_slider(8.,-12.,12.,'Base u',1.),
    'base_v':core_slider(4.,-8.,8.,'Base v',1.),
    'roundtrip':W.Checkbox(value=False,description='Return to original grid')})

### Проверка на стенде 5

1. Для постоянного (8,4), width=.5 и height=1.5 вычислите ожидаемый (4,6). Сравните correct, none, same_for_both и swapped_axes, включая численную ошибку.
2. Поменяйте знаки исходных u/v и отдельно каждую ось масштаба. Почему один общий коэффициент не исправляет анизотропный resize?
3. Выберите two_motions и уменьшение .25 по обеим осям. Включите возврат в исходную сетку, сравните linear/nearest: какое различие связано с потерей границы, а какое — с единицами?
4. На rotation сравните увеличенную и уменьшенную сетки. Затем проверьте sx=sy=1: почему этот контрольный случай не заменяет проверки разных осей? Сохраните два сравнения с указанной сеткой и единицами EPE.

## 6. EPE: общая маска, сложные области и знаменатель

EPE — средняя евклидова норма ошибки по общей GT-valid маске: √((u−u*)²+(v−v*)²). Для ошибки (−.3,.4) получается .5 px. Стенд отдельно управляет допустимыми ошибками, сложной областью и значениями вне GT. Пустая маска имеет неопределённое среднее, NaN, а не нулевую ошибку.

Третья оценка намеренно выбирает пиксели по их истинной ошибке. Это оптимистичный диагностический пример: уменьшение среднего вместе с покрытием не является улучшением метода. Такая маска не может заменить общую GT-valid маску в сравнении алгоритмов.

In [ ]:
def show_mask(error_u,error_v,outlier,invalid_width,hotspot_fraction,invalid_nan,threshold,empty_mask):
    h,w=48,72;x,y=grid(h,w);true=np.zeros((h,w,2),np.float32);true[:]=[2,1]
    pred=true+np.array([error_u,error_v]);hotspot=x<round(w*hotspot_fraction)
    pred[hotspot,0]+=6
    valid=x<w-invalid_width
    if empty_mask:valid[:]=False
    pred[~valid]=np.nan if invalid_nan else true[~valid]+np.array([outlier,0])
    error=np.linalg.norm(pred-true,axis=2);right=core_mean(error,valid)
    wrong=float(np.mean(error))
    # Deliberately selective diagnostic; never replaces GT-valid scoring.
    selected=valid&(error<=threshold);selective=core_mean(error,selected)
    coverage=float(selected.sum()/valid.sum()) if valid.any() else float('nan')
    fig,axs=plt.subplots(2,2,figsize=(10.8,6),constrained_layout=True)
    im=axs[0,0].imshow(valid,cmap='gray',vmin=0,vmax=1);axs[0,0].set_title(f'GT-valid mask: {valid.sum()} pixels');axs[0,0].axis('off')
    im=axs[0,1].imshow(np.where(valid,error,np.nan),cmap='magma',vmin=0,vmax=8)
    axs[0,1].set_title('EPE; white is excluded');axs[0,1].axis('off');fig.colorbar(im,ax=axs[0,1],shrink=.8,label='px')
    axs[1,0].imshow(selected,cmap='gray',vmin=0,vmax=1);axs[1,0].set_title(f'Selective error ≤ {threshold:g}: coverage={coverage:.1%}');axs[1,0].axis('off')
    labels=['GT-valid','Wrong: all','Selective subset'];values=[right,wrong,selective]
    axs[1,1].bar(labels,[v if np.isfinite(v) else 0 for v in values],color=['seagreen','tomato','steelblue'])
    for j,value in enumerate(values):axs[1,1].text(j,(value if np.isfinite(value) else 0)+.05,f'{value:.3f}' if np.isfinite(value) else 'NaN / undefined',ha='center',fontsize=8)
    axs[1,1].set(title='Different denominators',ylabel='mean EPE, px',ylim=(0,max([1]+[v*1.25 for v in values if np.isfinite(v)])))
    plt.show();plt.close(fig)
    print(f'GT-valid EPE={right}; count={valid.sum()}; naive all-pixel mean={wrong}.')
    print(f'Selective mean={selective}; selected={selected.sum()}; coverage={coverage}. This subset is chosen using true error and is deliberately optimistic.')
    print('An empty GT mask has undefined mean (NaN), never zero. Arbitrary numbers or NaN outside GT-valid must not affect its score.')
    return {'epe':right,'naive':wrong,'count':int(valid.sum()),'selected_epe':selective,
            'coverage':coverage,'selected_count':int(selected.sum()),'hotspot_valid_count':int((hotspot&valid).sum())}
panel('mask',show_mask,{
    'error_u':core_slider(-.3,-2.,2.,'Valid u error',.1),
    'error_v':core_slider(.4,-2.,2.,'Valid v error',.1),
    'outlier':core_slider(10.,0.,30.,'Invalid error',5.),
    'invalid_width':core_slider(12,0,72,'Invalid strip width',6),
    'hotspot_fraction':core_slider(0.,0.,.5,'Hard region fraction',.1),
    'invalid_nan':W.Checkbox(value=False,description='NaN outside GT mask'),
    'threshold':core_slider(2.,0.,8.,'Selective EPE limit',.5),
    'empty_mask':W.Checkbox(value=False,description='Empty GT mask')})

### Проверка на стенде 6

1. При ошибке (−.3,.4) измените invalid error 10 → 30, затем включите NaN вне GT. Предскажите GT-valid EPE и naive mean; проверьте независимость правильной оценки от недопустимых значений.
2. Задайте ошибку (.6,.8), без сложной области. Получите вручную EPE; объясните, почему это 1, а не среднее модулей компонент.
3. Добавьте hard region fraction=.3 и понижайте selective limit. Сопоставьте среднее, число пикселей и coverage. Почему это не результат нового более точного метода?
4. Меняйте ширину invalid-полосы, включая всё изображение; отдельно включите Empty GT mask. Какое значение означает «ответ не определён»? Запишите минимум два опыта, явно указав маску и знаменатель.

## 7. Farnebäck и DIS: общий вход, разная механика

Farnebäck приближает локальную интенсивность полиномом и уточняет плотное поле по уровням; размер окна влияет на поддержку и смешивание движений. DIS использует быстрый поиск соответствий патчей, распространение и вариационное уточнение. Здесь запускаются **реальные OpenCV-оценщики**, управляются их параметры, а вход и GT-valid маска общие.

Изображения могут уменьшаться перед оценкой. После возврата поля в исходную сетку компоненты u/v масштабируются по соответствующим осям, поэтому EPE остается в исходных пикселях. Общее изменение яркости нарушает исходное постоянство интенсивности; относительная устойчивость зависит и от механизма, и от настроек. Скорость показана как единичный локальный замер полного вызова оценки, включая создание DIS и изменения разрешения.

In [ ]:
def show_dense(motion, dx, scale, fb_levels, fb_window, dis_preset, dis_refine, brightness):
    a, b, truth, valid, boundary = adv_pair(dx, kind=motion, brightness=brightness)
    h, w = a.shape
    flows, timings = [], []
    for method in ['Farneback', 'DIS']:
        begin = time.perf_counter()
        if scale == 1:
            aa, bb = a, b
        else:
            size = (round(w*scale), round(h*scale))
            aa = cv2.resize(a, size, interpolation=cv2.INTER_AREA)
            bb = cv2.resize(b, size, interpolation=cv2.INTER_AREA)
        if method == 'Farneback':
            flow = cv2.calcOpticalFlowFarneback(aa, bb, None, .5, fb_levels, fb_window, 5, 5, 1.2, 0)
        else:
            solver = cv2.DISOpticalFlow_create(int(dis_preset))
            solver.setVariationalRefinementIterations(dis_refine)
            flow = solver.calc(aa, bb, None)
        if scale != 1:
            flow = adv_resize_flow(flow, (h, w))
        timings.append(1000*(time.perf_counter()-begin)); flows.append(flow)
    errors = [np.linalg.norm(flow-truth, axis=2) for flow in flows]
    vmax = max(4, abs(dx)+2)
    emax = max(1., max(float(np.percentile(err[valid], 95)) for err in errors))
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
    for ax, image, title in zip(axes.flat, [a, b, flow_rgb(truth, vmax), valid], ['I1', f'I2; brightness +{brightness}', 'GT flow; shared direction colors', 'Common GT-valid for both methods']):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255 if image.dtype == np.uint8 else 1); ax.set_title(title); ax.axis('off')
    plt.show(); plt.close(fig)
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
    for k, (method, flow, error) in enumerate(zip(['Farneback', 'DIS'], flows, errors)):
        axes[k, 0].imshow(flow_rgb(flow, vmax)); axes[k, 0].set_title(f'{method}: {timings[k]:.2f} ms'); axes[k, 0].axis('off')
        im = axes[k, 1].imshow(np.where(valid, error, np.nan), cmap='magma', vmin=0, vmax=emax)
        axes[k, 1].set_title(f'EPE={adv_mean(error, valid):.3f} px'); axes[k, 1].axis('off')
    fig.colorbar(im, ax=axes[:, 1], shrink=.75, label='Endpoint error, original px; common scale')
    plt.show(); plt.close(fig)
    result = {'farneback_epe': adv_mean(errors[0], valid), 'dis_epe': adv_mean(errors[1], valid),
        'farneback_boundary_epe': adv_mean(errors[0], valid & boundary), 'dis_boundary_epe': adv_mean(errors[1], valid & boundary),
        'farneback_ms': timings[0], 'dis_ms': timings[1], 'valid_count': int(valid.sum()),
        'input_shape': [round(h*scale), round(w*scale)], 'returned_shape': list(truth.shape),
        'farneback_mean_u': adv_mean(flows[0][..., 0], valid), 'dis_mean_u': adv_mean(flows[1][..., 0], valid)}
    print(result)
    print('Два настоящих оценщика, одна пара и одна маска. Время: создание оценщика + resize входа + расчет + возврат векторов; без синтетики и графиков, один CPU thread. Единичный локальный замер шумный; общего рейтинга методов он не задает.')
    print('Эти настройки служат исследованию на учебных парах; не переносите подбор на фиксированный test основного задания.')
    return result

panel('dense', show_dense, {
    'motion': W.Dropdown(options=[('Общий перенос', 'translation'), ('Движущаяся вставка', 'moving_patch')], value='translation', description='Сцена'),
    'dx': W.IntSlider(value=6, min=-16, max=16, description='Сдвиг x'),
    'scale': W.SelectionSlider(options=[.5, .75, 1.], value=1., description='Масштаб'),
    'fb_levels': W.IntSlider(value=3, min=1, max=5, description='FB уровни'),
    'fb_window': W.SelectionSlider(options=[5, 9, 15, 25, 35], value=15, description='FB окно'),
    'dis_preset': W.Dropdown(options=[('Ultrafast', 0), ('Fast', 1), ('Medium', 2)], value=2, description='DIS preset'),
    'dis_refine': W.IntSlider(value=5, min=0, max=10, description='DIS refine'),
    'brightness': W.IntSlider(value=0, min=-50, max=50, description='Яркость I2')})

### Проверка на стенде 7

1. На общем переносе запишите EPE обоих методов и время; затем **без других изменений** уменьшите масштаб 1 → .5. Проверьте, что поле возвращено в исходную форму и EPE измерена в тех же пикселях.
2. Переключите сцену на движущуюся вставку. Измените только окно FB 5 → 35. Сравните общую и boundary EPE: почему уменьшение шума может сопровождаться размазыванием границы?
3. Зафиксируйте сцену и смените DIS preset, затем число variational refinement 0 → 10. Запишите отрицательный результат, если прирост качества отсутствует; повторите замер времени несколько раз.
4. Задайте изменение яркости −50 или +50. Сравните EPE с исходным опытом и видимое изменение изображений; для фотометрического residual вернитесь к стенду 1. Почему ошибка изображения и ошибка геометрии могут вести себя по-разному? Эти опыты — учебная настройка, параметры фиксированного test основного ноутбука не менять.

## 8. Forward/backward consistency: координаты и ложная уверенность

Для видимого соответствия проверяется $r(x)=\|f_{12}(x)+f_{21}(x+f_{12}(x))\|$. Прямое поле задано на I1, обратное — на I2. При расширении изображения обратный вектор меняется по позиции; сложение массивов без семплирования дает другой ответ. Здесь используются аналитически известные affine-поля, к которым добавляются управляемые ошибки.

Прибытие в полосу нового перекрытия означает отсутствие соответствия. GT-маска переносит эту полосу с I2 на I1 по **истинному** полю; FB-фильтр работает по оцененным полям. Покрытие и EPE после фильтрации показаны отдельно от основной EPE на общей GT-области. Два поля могут согласованно описывать неверное соответствие.

In [ ]:
def show_consistency(dx, expansion, forward_bias, backward_bias, threshold, occlusion, sampling, coupled):
    h, w = 64, 96; x, y = grid(h, w)
    cx, cy = (w-1)/2, (h-1)/2; scale = 1+expansion
    truth = np.dstack([dx+expansion*(x-cx), expansion*(y-cy)]).astype(np.float32)
    backward = np.dstack([((x-cx-dx)/scale+cx)-x, ((y-cy)/scale+cy)-y]).astype(np.float32)
    forward = truth.copy(); forward[..., 0] += forward_bias
    if coupled:
        # Exact inverse of the biased affine forward map. Agreement can still be wrong.
        backward[..., 0] -= forward_bias/scale
    backward[..., 0] += backward_bias
    occluded_target = (x >= w//2-occlusion/2) & (x < w//2+occlusion/2)
    backward[occluded_target] = [0, 0]
    sampled, inside = adv_warp(backward, forward)
    occluded_at_gt, truth_inside = adv_warp(occluded_target.astype(np.float32), truth)
    gt_valid = truth_inside & (occluded_at_gt < .01)
    if sampling == 'sample':
        chosen = sampled
    else:
        chosen = backward
    residual = np.linalg.norm(forward+chosen, axis=2)
    accepted = inside & (residual <= threshold)
    error = np.linalg.norm(forward-truth, axis=2)
    good_denominator = int(gt_valid.sum())
    selected = gt_valid & accepted
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)
    images = [truth[..., 0], forward[..., 0], backward[..., 0], np.where(inside, residual, np.nan), accepted, gt_valid]
    titles = ['GT forward u on I1', 'Estimated forward u on I1', 'Backward u on I2', 'FB residual after selected operation', 'Accepted by FB threshold', 'GT-visible source pixels']
    for i, (ax, image, title) in enumerate(zip(axes.flat, images, titles)):
        if i < 3:
            im = ax.imshow(image, cmap='coolwarm', vmin=-max(2, abs(dx)+abs(expansion)*w/2+abs(forward_bias)+abs(backward_bias)), vmax=max(2, abs(dx)+abs(expansion)*w/2+abs(forward_bias)+abs(backward_bias)))
        elif i == 3:
            im = ax.imshow(image, cmap='magma', vmin=0, vmax=max(1, threshold*2, float(np.nanpercentile(image, 95))))
        else:
            im = ax.imshow(image, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title); ax.axis('off')
        if i in (0, 3): fig.colorbar(im, ax=ax, shrink=.7, label='pixels')
    plt.show(); plt.close(fig)
    result = {'residual_mean': adv_mean(residual, inside), 'coverage': float(selected.sum()/good_denominator) if good_denominator else float('nan'),
        'full_gt_epe': adv_mean(error, gt_valid), 'selected_epe': adv_mean(error, selected), 'valid_count': good_denominator,
        'accepted_count': int(selected.sum()), 'inside_count': int(inside.sum()),
        'backward_sample_difference': adv_mean(np.linalg.norm(sampled-backward, axis=2), inside), 'forward_mean_u': float(forward[..., 0].mean())}
    print(result)
    print('Основная EPE использует общую GT-valid. Coverage — доля этой области, принятая FB. Отдельная EPE на принятой части меняет знаменатель. Низкий FB residual не доказывает правильность двух согласованно ошибочных полей.')
    print('Это аналитически заданные affine-поля с контролируемой ошибкой, не выход обученной сети. Полоса перекрытия задана на I2, ее GT-маска перенесена на I1.')
    return result

panel('consistency', show_consistency, {
    'dx': W.IntSlider(value=3, min=-10, max=10, description='Перенос x'),
    'expansion': W.FloatSlider(value=.25, min=-.25, max=.5, step=.05, description='Расширение'),
    'forward_bias': W.FloatSlider(value=0., min=-3, max=3, step=.25, description='Ошибка f12'),
    'backward_bias': W.FloatSlider(value=0., min=-3, max=3, step=.25, description='Ошибка f21'),
    'threshold': W.FloatSlider(value=.15, min=0, max=4, step=.05, description='Порог FB'),
    'occlusion': W.IntSlider(value=0, min=0, max=24, step=2, description='Полоса I2'),
    'sampling': W.Dropdown(options=[('Семплировать f21', 'sample'), ('Ошибочно сложить сетки', 'pointwise')], value='sample', description='Операция'),
    'coupled': W.Checkbox(value=False, description='Согласовать ошибки')})

### Проверка на стенде 8

1. При нулевых ошибках сравните «Семплировать f21» и ошибочное сложение сеток. Затем задайте расширение=0: почему постоянный перенос плохо обнаруживает эту ошибку реализации?
2. Добавьте ошибку f12=2, оставив ошибку f21=0. Изменяйте порог FB и записывайте coverage вместе с полной GT EPE. Меняет ли порог само поле?
3. С той же ошибкой f12 включите «Согласовать ошибки». Найдите низкий residual при ненулевой EPE. Почему согласованность — диагностический признак, а не гарантия правильности?
4. Включите полосу перекрытия I2. Сравните ее положение в GT-valid на I1 и принятое FB подмножество. Объясните, почему нельзя заменить общую маску метода его собственным удобным подмножеством.

## 9. Поток как инструмент: перенос и агрегация соседнего кадра

Поток можно **использовать** для переноса соседнего изображения или его признаков. Здесь очищается зашумленный кадр: сосед сначала приводится на сетку опорного, затем изображения смешиваются с весом $a$. Чистый опорный кадр известен только благодаря синтетике и позволяет вычислить настоящую ошибку изображения.

При точном целочисленном переносе и независимом шуме усреднение уменьшает шум. Ошибочный поток создает двоение, а новый заслоняющий объект переносит чужое содержимое. Управляемая маска в этом стенде использует **известную** синтетическую видимость; это полезная верхняя оценка, а не готовое решение видимости в реальном видео. Стенд иллюстрирует идею потоко-управляемой агрегации, **не реализует BasicVSR или FGFA**.

In [ ]:
def show_transfer(dx, dy, noise, alpha, flow_bias, occlusion, use_mask, alignment):
    h, w = 96, 144; clean = adv_image(81, h, w).astype(np.float32)
    # A sharp feature makes ghosting and edge losses visible in addition to noise.
    clean[42:48, 70:105] = 235
    rng = np.random.default_rng(82)
    reference = np.clip(clean + rng.normal(0, noise, clean.shape), 0, 255).astype(np.float32)
    neighbor_clean = cv2.warpAffine(clean, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h))
    neighbor = np.clip(neighbor_clean + rng.normal(0, noise, clean.shape), 0, 255).astype(np.float32)
    x, y = grid(h, w)
    covered = (x >= w//2-occlusion/2) & (x < w//2+occlusion/2)
    neighbor[covered] = 30  # New foreground has no counterpart in the reference.
    truth = np.zeros((h, w, 2), np.float32); truth[:] = [dx, dy]
    field = truth.copy(); field[..., 0] += flow_bias
    if not alignment:
        field[:] = 0
    aligned, inside = adv_warp(neighbor, field)
    sampled_occlusion, _ = adv_warp(covered.astype(np.float32), field)
    # Oracle visibility is deliberately known in this synthetic teaching example.
    weight = np.full((h, w), alpha, np.float32)
    if use_mask:
        weight *= inside & (sampled_occlusion < .01)
    fused = (1-weight)*reference + weight*aligned
    truth_occlusion, truth_inside = adv_warp(covered.astype(np.float32), truth)
    visible = truth_inside & (truth_occlusion < .01)
    base_error = np.abs(reference-clean); fused_error = np.abs(fused-clean)
    edge = np.abs(cv2.Sobel(clean, cv2.CV_32F, 1, 0, ksize=3)) + np.abs(cv2.Sobel(clean, cv2.CV_32F, 0, 1, ksize=3)) > 150
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)
    for ax, image, title in zip(axes.flat, [clean, reference, neighbor, aligned, fused, fused_error],
            ['Clean reference (known target)', 'Noisy reference I1', 'Noisy neighbor I2 + occluder', 'I2 sampled on reference grid', 'Weighted temporal aggregation', 'Absolute image error vs clean I1']):
        im = ax.imshow(image, cmap='gray' if ax is not axes[1, 2] else 'magma', vmin=0, vmax=255 if ax is not axes[1, 2] else max(10, float(np.percentile(fused_error, 95))))
        ax.set_title(title); ax.axis('off')
    fig.colorbar(im, ax=axes[1, 2], shrink=.8, label='Image intensity error (0–255)')
    plt.show(); plt.close(fig)
    result = {'reference_mae': float(base_error.mean()), 'fused_mae': float(fused_error.mean()),
        'visible_reference_mae': adv_mean(base_error, visible), 'visible_fused_mae': adv_mean(fused_error, visible),
        'edge_mae': adv_mean(fused_error, edge), 'used_neighbor_fraction': float((weight > 0).mean()),
        'mean_neighbor_weight': float(weight.mean()), 'gt_visible_count': int(visible.sum()),
        'reference_noise_rms': float(np.sqrt(np.mean((reference-clean)**2))), 'effective_flow': [float(field[0,0,0]), float(field[0,0,1])]}
    print(result)
    print('Учебная агрегация двух изображений: (1−a)·I1 + a·warp(I2). Это демонстрация использования потока, не реализация BasicVSR/FGFA и не обучение. Значения MAE относятся к изображению, а не к EPE потока.')
    print('Маска здесь известна из синтетики: подавляет выход за кадр и новую заслоняющую полосу. В реальном видео надежную видимость еще нужно оценить. При a=0 сосед не используется.')
    return result

panel('transfer', show_transfer, {
    'dx': W.IntSlider(value=6, min=-16, max=16, description='Сдвиг x'),
    'dy': W.IntSlider(value=2, min=-8, max=8, description='Сдвиг y'),
    'noise': W.IntSlider(value=20, min=0, max=40, description='Шум кадров'),
    'alpha': W.FloatSlider(value=.5, min=0, max=1, step=.1, description='Вес соседа'),
    'flow_bias': W.FloatSlider(value=0., min=-4, max=4, step=.5, description='Ошибка u'),
    'occlusion': W.IntSlider(value=0, min=0, max=28, step=2, description='Перекрытие'),
    'use_mask': W.Checkbox(value=True, description='Маска видимости'),
    'alignment': W.Checkbox(value=True, description='Выравнивать соседа')})

### Проверка на стенде 9

1. При шуме=20, точном потоке и маске сравните вес соседа 0 → .5 → 1. Запишите MAE изображения и объясните, почему смесь двух независимых наблюдений может оказаться лучше каждого по отдельности.
2. Отключите выравнивание при ненулевом переносе; затем включите его и добавьте ошибку u=2. Сравните MAE на границах и видимый результат: где вместо подавления шума появилось двоение?
3. Добавьте полосу перекрытия и сравните маску видимости в состояниях включено/выключено. Сопоставьте долю реально использованного соседа с общей MAE.
4. Установите шум=0 и вес=.5. При правильном выравнивании с маской изображение восстанавливается точно; при ошибке потока оно ухудшается. Почему выигрыш «по красоте» не заменяет проверку против чистого целевого кадра?

## Самостоятельный эксперимент
В `02_flow_experiment.ipynb` реализуйте warping, resize, masked EPE, LK и
forward/backward consistency, затем сравните Farnebäck и DIS на общих парах.
Настройки этого демо не меняют фиксированный протокол основной работы:
выбор по val, затем фиксированный test и заранее объявленная абляция.

Источники: [OpenCV: optical flow](https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html),
[параметры LK и Farnebäck](https://docs.opencv.org/4.x/dc/d6b/group__video__track.html),
[Horn–Schunck](https://people.csail.mit.edu/bkph/papers/Optical_Flow_OPT.pdf),
[DIS](https://arxiv.org/abs/1603.03590).